# Dana's Bakery — deterministic LangGraph orders
## 1. Project overview
Process independent bakery orders with validation, up to two substitutions, precise pricing, delivery, and friendly messages. No models, keys, or credentials are needed.

**Prerequisites:** a fresh Google Colab Python runtime and internet for package installation and the Mermaid PNG renderer. Upload this notebook, then select **Runtime → Run all**. Inventory is a fixed snapshot per invocation.

**Expected output:** the compiled graph image, the example order totaling 80.00 NIS, and a passing test summary. All implementation and tests are contained in notebook cells.

## 2. Dependency installation

In [ ]:
%pip install -q "langgraph==1.2.20"

## 3. Imports

In [ ]:
from decimal import Decimal, ROUND_HALF_UP
from typing import Any, TypedDict
from IPython.display import Image, display
from langgraph.graph import START, END, StateGraph

## 4. Product catalog and inventory

In [ ]:
CATALOG = {
    "challah": {"price": Decimal("18"), "stock": 10},
    "babka": {"price": Decimal("45"), "stock": 0},
    "rugelach": {"price": Decimal("30"), "stock": 3},
    "sourdough": {"price": Decimal("25"), "stock": 4},
}
SUBSTITUTES = {"babka": "rugelach", "rugelach": "sourdough", "sourdough": "challah"}
WEEKDAYS = {"monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"}
ZERO = Decimal("0.00")


def money(amount: Decimal) -> Decimal:
    return amount.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)

## 5. State definition
Input annotations allow malformed values so the validation node can reject them politely. Monetary state and totals use `Decimal`.

In [ ]:
class OrderState(TypedDict, total=False):
    item: Any
    quantity: Any
    day: Any
    city: Any
    original_item: str | None
    substitution_attempts: int
    subtotal: Decimal
    discount: Decimal
    delivery_fee: Decimal
    rejection_reason: str
    status: str
    delivery: str
    total: Decimal
    message: str
    steps: list[str]
    valid: bool
    stock_available: bool
    candidate: str | None

## 6. Node implementations
Each node appends its name. Each actual substitution also adds a detailed attempt entry. Validation replaces all internal fields, including history, on every invocation.

In [ ]:
def normalize(value: str) -> str:
    return " ".join(value.split()).casefold()


def validate_order(state: OrderState) -> dict:
    item, quantity = state.get("item"), state.get("quantity")
    day, city = state.get("day"), state.get("city")
    item = normalize(item) if isinstance(item, str) else None
    day = normalize(day) if isinstance(day, str) else None
    normalized_city = normalize(city) if isinstance(city, str) else None
    errors = []
    if item not in CATALOG:
        errors.append("please choose challah, babka, rugelach, or sourdough; croissants are discontinued")
    if type(quantity) is not int or quantity <= 0:
        errors.append("quantity must be a positive integer (not a boolean)")
    if day not in WEEKDAYS:
        errors.append("please provide a valid English weekday")
    if city is not None and not isinstance(city, str):
        errors.append("city must be text or None")
    return {
        "item": item, "quantity": quantity, "day": day, "city": normalized_city,
        "original_item": item, "candidate": item, "substitution_attempts": 0,
        "subtotal": ZERO, "discount": ZERO, "delivery_fee": ZERO,
        "rejection_reason": "; ".join(errors), "valid": not errors,
        "stock_available": False, "status": "rejected", "delivery": "pickup",
        "total": ZERO, "message": "", "steps": ["validate_order"],
    }


def check_stock(state: OrderState) -> dict:
    return {
        "stock_available": CATALOG[state["item"]]["stock"] >= state["quantity"],
        "steps": state["steps"] + ["check_stock"],
    }


def try_substitute(state: OrderState) -> dict:
    steps = state["steps"] + ["try_substitute"]
    candidate = SUBSTITUTES.get(state["candidate"])
    if candidate is None:
        return {"stock_available": False, "rejection_reason": "insufficient stock and no further substitute is available", "steps": steps}
    attempts = state["substitution_attempts"] + 1
    available = CATALOG[candidate]["stock"] >= state["quantity"]
    steps.append(f"substitution_attempt_{attempts}: {state['candidate']} -> {candidate} ({'available' if available else 'insufficient stock'})")
    return {
        "candidate": candidate, "item": candidate,
        "substitution_attempts": attempts, "stock_available": available,
        "rejection_reason": (f"insufficient stock after {attempts} substitution attempt(s); no further eligible substitute is available") if not available else "",
        "steps": steps,
    }


def calculate_price(state: OrderState) -> dict:
    subtotal = CATALOG[state["item"]]["price"] * state["quantity"]
    discount = money(subtotal * Decimal("0.15")) if subtotal > Decimal("150") and state["day"] != "friday" else ZERO
    return {"status": "accepted", "subtotal": money(subtotal), "discount": discount,
            "total": money(subtotal - discount), "steps": state["steps"] + ["calculate_price"]}


def handle_delivery(state: OrderState) -> dict:
    delivery = "delivery" if state["city"] == "jerusalem" else "pickup"
    fee = Decimal("20.00") if delivery == "delivery" else ZERO
    return {"delivery": delivery, "delivery_fee": fee, "total": money(state["total"] + fee),
            "steps": state["steps"] + ["handle_delivery"]}


def reject_order(state: OrderState) -> dict:
    return {"status": "rejected", "item": None, "delivery": "pickup", "total": ZERO,
            "steps": state["steps"] + ["reject_order"]}


def generate_message(state: OrderState) -> dict:
    if state["status"] == "rejected":
        message = f"We're sorry, we couldn't complete your order: {state['rejection_reason']}. Thank you for understanding!"
    else:
        message = f"Thank you for ordering from Dana's Bakery! Your order of {state['quantity']} × {state['item']} is accepted. "
        if state["item"] != state["original_item"]:
            message += f"We automatically replaced {state['original_item']} with {state['item']} due to limited stock. "
        if state["delivery"] == "delivery":
            message += "Delivery to Jerusalem includes a 20.00 NIS delivery fee. "
        elif state["city"]:
            message += "Delivery is available only in Jerusalem, so your order has been changed to pickup. "
        else:
            message += "Your order is ready for pickup. "
        if state["discount"]:
            message += f"Your baked goods include a 15% discount of {state['discount']:.2f} NIS. "
        message += f"Total: {state['total']:.2f} NIS. We look forward to seeing you!"
    return {"message": message, "steps": state["steps"] + ["generate_message"]}

## 7. Conditional routing functions
The substitution counter increases only for actual candidates. Every loop consumes one attempt, so all paths terminate.

In [ ]:
def route_validation(state: OrderState) -> str:
    return "valid" if state["valid"] else "invalid"


def route_stock(state: OrderState) -> str:
    return "available" if state["stock_available"] else "unavailable"


def route_substitution(state: OrderState) -> str:
    if state["stock_available"]:
        return "found"
    if state["substitution_attempts"] < 2 and state["candidate"] in SUBSTITUTES:
        return "retry"
    return "reject"

## 8. Graph construction and compilation

In [ ]:
builder = StateGraph(OrderState)
for node in (validate_order, check_stock, try_substitute, calculate_price,
             handle_delivery, reject_order, generate_message):
    builder.add_node(node.__name__, node)
builder.add_edge(START, "validate_order")
builder.add_conditional_edges("validate_order", route_validation,
                              {"valid": "check_stock", "invalid": "reject_order"})
builder.add_conditional_edges("check_stock", route_stock,
                              {"available": "calculate_price", "unavailable": "try_substitute"})
builder.add_conditional_edges("try_substitute", route_substitution,
                              {"found": "calculate_price", "retry": "try_substitute", "reject": "reject_order"})
builder.add_edge("calculate_price", "handle_delivery")
builder.add_edge("handle_delivery", "generate_message")
builder.add_edge("reject_order", "generate_message")
builder.add_edge("generate_message", END)
graph = builder.compile()

## 9. Mermaid graph visualization
This renders the actual compiled graph, including its substitution loop. The default renderer requires internet access to Mermaid.

In [ ]:
display(Image(graph.get_graph().draw_mermaid_png()))

## 10. Test orders and assertions
Expected totals: original 36; first substitute + delivery 80; second substitute 100; discounted pickup 153; Friday 180; outside Jerusalem pickup 36; discounted delivery 173. Invalid and exhausted-stock orders return zero. The exact 150 NIS threshold cannot occur with this fixed catalog and stock; a direct pricing-node boundary check isolates that rule without changing inventory.

Test runner: notebook assertions. In Colab select **Runtime → Run all**. Local equivalent: `python -m nbclient danas_bakery.ipynb --timeout=180`.

In [ ]:
result = graph.invoke({
    "item": "babka", "quantity": 2, "day": "Thursday", "city": "Jerusalem"
})
print(result["message"])

REQUIRED = {"status", "item", "delivery", "total", "message", "steps"}
ACCEPT_TAIL = ["calculate_price", "handle_delivery", "generate_message"]
REJECT_TAIL = ["reject_order", "generate_message"]


def order(item="challah", quantity=2, day="Thursday", city=None):
    return {"item": item, "quantity": quantity, "day": day, "city": city}


def path(attempts=(), invalid=False, rejected=False, no_substitute=False):
    expected = ["validate_order"]
    if not invalid:
        expected.append("check_stock")
        for number, (source, target, available) in enumerate(attempts, 1):
            expected += ["try_substitute", f"substitution_attempt_{number}: {source} -> {target} ({'available' if available else 'insufficient stock'})"]
        if no_substitute:
            expected.append("try_substitute")
    return expected + (REJECT_TAIL if rejected else ACCEPT_TAIL)


# name, input, final item, total, delivery, expected path, message fragment
cases = [
    ("original", order(), "challah", "36", "pickup", path(), "pickup"),
    ("first substitute", order("babka", city="Jerusalem"), "rugelach", "80", "delivery", path([("babka", "rugelach", True)]), "replaced babka with rugelach"),
    ("second substitute", order("babka", 4), "sourdough", "100", "pickup", path([("babka", "rugelach", False), ("rugelach", "sourdough", True)]), "replaced babka with sourdough"),
    ("two failed attempts", order("babka", 5), None, "0", "pickup", path([("babka", "rugelach", False), ("rugelach", "sourdough", False)], rejected=True), "insufficient stock"),
    ("invalid product", order("pizza"), None, "0", "pickup", path(invalid=True, rejected=True), "please choose"),
    ("zero quantity", order(quantity=0), None, "0", "pickup", path(invalid=True, rejected=True), "positive integer"),
    ("discount", order(quantity=10), "challah", "153", "pickup", path(), "15%"),
    ("Friday exclusion", order(quantity=10, day="Friday"), "challah", "180", "pickup", path(), "180.00"),
    ("outside Jerusalem", order(city="Tel Aviv"), "challah", "36", "pickup", path(), "changed to pickup"),
    ("discontinued", order("croissant"), None, "0", "pickup", path(invalid=True, rejected=True), "discontinued"),
    ("discount excludes fee", order(quantity=10, city="Jerusalem"), "challah", "173", "delivery", path(), "20.00"),
    ("no substitute", order(quantity=11), None, "0", "pickup", path(rejected=True, no_substitute=True), "no further substitute"),
    ("normalization", order("  CHALLAH ", day=" thURsday ", city="  JERUSALEM  "), "challah", "56", "delivery", path(), "Delivery to Jerusalem"),
    ("blank city", order(city="  "), "challah", "36", "pickup", path(), "pickup"),
    ("rugelach substitution", order("rugelach", 5), "challah", "90", "pickup", path([("rugelach", "sourdough", False), ("sourdough", "challah", True)]), "replaced rugelach with challah"),
    ("chain ends after one failure", order("sourdough", 11), None, "0", "pickup", path([("sourdough", "challah", False)], rejected=True), "insufficient stock"),
]
for field, value in [("quantity", -1), ("quantity", True), ("quantity", 2.0),
                     ("quantity", "2"), ("quantity", None), ("item", []),
                     ("item", None), ("day", "Funday"), ("day", 4),
                     ("city", 123), ("city", []), ("day", None)]:
    malformed = order()
    malformed[field] = value
    cases.append((f"malformed {field}={value!r}", malformed, None, "0", "pickup",
                  path(invalid=True, rejected=True), "sorry"))
cases.append(("missing inputs", {}, None, "0", "pickup", path(invalid=True, rejected=True), "sorry"))

for name, inputs, item, total, delivery, steps, fragment in cases:
    actual = graph.invoke(inputs)
    assert REQUIRED <= actual.keys(), name
    assert actual["status"] == ("accepted" if item else "rejected"), name
    assert actual["item"] == item, name
    assert actual["total"] == Decimal(total), name
    assert actual["delivery"] == delivery, name
    assert actual["steps"] == steps, (name, actual["steps"])
    visited = [node for update in graph.stream(inputs, stream_mode="updates") for node in update]
    assert visited == [entry for entry in steps if not entry.startswith("substitution_attempt_")], name
    assert 0 <= actual["substitution_attempts"] <= 2, name
    assert fragment.casefold() in actual["message"].casefold(), name
    if item:
        assert str(inputs["quantity"]) in actual["message"], name
        assert item in actual["message"], name
        assert f"{actual['total']:.2f}" in actual["message"], name
        assert delivery in actual["message"], name
        assert actual["quantity"] == inputs["quantity"], name
    print(f"PASS: {name}: {actual['status']}, {actual['total']:.2f} NIS")

# Isolate pricing at precisely 150 (not reachable within snapshot stock).
boundary = calculate_price({"item": "rugelach", "quantity": 5, "day": "thursday", "steps": []})
assert boundary["subtotal"] == Decimal("150.00")
assert boundary["discount"] == ZERO
assert boundary["total"] == Decimal("150.00")

# Repeated calls neither consume inventory nor reuse steps, even with stale internal input.
first = graph.invoke(order(quantity=10))
second = graph.invoke(order(quantity=10))
assert REQUIRED <= first.keys() and REQUIRED <= second.keys()
assert first == second
assert first["steps"] is not second["steps"]
stale = graph.invoke({**first, **order("babka", 2, city="Jerusalem")})
assert REQUIRED <= stale.keys()
assert stale["total"] == Decimal("80.00")
assert stale["steps"] == path([("babka", "rugelach", True)])
assert CATALOG["challah"]["stock"] == 10
print(f"All {len(cases)} order cases, boundary pricing, and repeated-invocation checks passed.")

## 11. Summary of results
After a successful Run all, the final output confirms all order cases, boundary pricing, and repeated-invocation checks passed. Tests assert all six output keys, exact paths and totals, bounded substitutions, message details, normalization, invalid types, and fresh history.

The graph has no persistent inventory or checkpoint memory. Automatic substitutions preserve quantity, and the delivery fee is added after the baked-goods discount.

LangGraph API reference: [StateGraph](https://reference.langchain.com/python/langgraph/graph/state/StateGraph) and [conditional edges](https://reference.langchain.com/python/langgraph/graph/state/StateGraph/add_conditional_edges).

**Local verification:** all code cells passed Python 3.12 syntax parsing; direct malformed-input validation and exact-150 pricing checks passed. Full LangGraph execution and PNG rendering were not completed locally because dependency downloads stalled. The full notebook test suite remains to be run in a fresh Colab runtime; no full-suite success is claimed.